In [0]:
%sql
SELECT *
FROM (
    SELECT
        *,
        
    FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cartentries`as a
    inner join delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_carts`as  b
    INNER JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentinfos`as c
    ON b.p_paymentinfo = c.PK and a.p_order = b.pk
    
) t
WHERE qtd_itens_no_carrinho > 2;

In [0]:
%sql
SELECT *
FROM (
    SELECT
        *,
        COUNT(p_order) OVER (PARTITION BY p_order) AS qtd_itens_no_carrinho
    FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cartentries`as a
    inner join delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_carts`as  b
    INNER JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentinfos`as c
    ON b.p_paymentinfo = c.PK and a.p_order = b.pk
    
) t
WHERE qtd_itens_no_carrinho > 2;


In [0]:
%sql
SELECT *
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_carts`
WHERE p_paymentstatus is not null


In [0]:
%sql
WITH base_carts AS (
  SELECT
    c.*,
    CASE
      WHEN pi.PK IS NULL THEN 'abandoned'
      ELSE 'completed'
    END AS cart_status
  FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_carts` c
  LEFT JOIN delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentinfos` pi
    ON c.p_paymentinfo = pi.PK
)
SELECT *
FROM base_carts
where cart_status =  'completed'
and pk = 9710104608811
limit 1



In [0]:
%sql
SELECT *
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_carts`
WHERE p_site = 8796093056040

In [0]:
%sql
SELECT *
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_paymentinfos`
WHERE pk = 8891313815594


In [0]:
%sql
SELECT *
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cartentries`
WHERE p_order = 9710104608811

In [0]:
%sql
SELECT *
FROM delta.`dbfs:/Volumes/workspace/cantustore/silver_prova_dados/tb_cmssitelp`
WHERE itempk = 8801173012481


In [0]:
# Caminho de saída do TXT (pode ajustar o nome/pasta se quiser)
OUT_TXT = "dbfs:/Volumes/workspace/cantustore/gold_prova_dados/top50_abandoned_carts.txt"
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/cantustore/gold_prova_dados")

# Coleta das linhas já prontas
lines = [r["line"] for r in spark.sql("SELECT line FROM vw_top50_abandoned_txt").collect()]

# Escrita em arquivo único
content = "\n".join(lines) + "\n"
dbutils.fs.put(OUT_TXT, content, overwrite=True)

print("TXT gerado em:", OUT_TXT)
